# Linguistic specification statistics

Summarizes the per-example linguistic statistics for RefEdit-Bench Easy/Hard and CompBench Local/Multi-object annotations. Each row is the unweighted mean over available annotation records in that group; reduction is the mean of the per-example reductions.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from framework.utils.project import find_project_root

OUTPUTS = find_project_root() / ".out/annotation"
REFEDIT_BENCH_PATH = OUTPUTS / "refedit_bench/refedit_bench_mixed.jsonl"
COMPBENCH_PATH = OUTPUTS / "compbench/compbench_local_multiobject_mixed.jsonl"

def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]

refedit_bench_records = read_jsonl(REFEDIT_BENCH_PATH)
compbench_records = read_jsonl(COMPBENCH_PATH)
print(f"Loaded {len(refedit_bench_records)} RefEdit-Bench and {len(compbench_records)} CompBench records")

In [ ]:
METRICS = [
    "original_words",
    "mixed_words",
    "reference_words_removed",
    "reduction",
]

rows = []
for record in refedit_bench_records:
    split = str(record.get("split", "")).casefold()
    if split in {"easy", "hard"}:
        rows.append({"Benchmark": f"RefEdit-Bench {split.title()}", **record["linguistic_stats"]})

for record in compbench_records:
    task = record.get("metadata", {}).get("task")
    if task in {"add", "remove", "replace"}:
        benchmark = "CompBench Local"
    elif task in {"multi_object_add", "multi_object_remove"}:
        benchmark = "CompBench Multi-object"
    else:
        continue
    rows.append({"Benchmark": benchmark, **record["linguistic_stats"]})

data = pd.DataFrame(rows)
if data.empty:
    raise ValueError("No matching annotation records were found")
missing = set(METRICS) - set(data.columns)
if missing:
    raise ValueError(f"Annotation records are missing linguistic metrics: {sorted(missing)}")

summary = data.groupby("Benchmark", sort=False)[METRICS].mean()
benchmark_order = [
    "RefEdit-Bench Easy",
    "RefEdit-Bench Hard",
    "CompBench Local",
    "CompBench Multi-object",
]
summary = summary.reindex([name for name in benchmark_order if name in summary.index])
summary = summary.rename(columns={
    "original_words": "Original words ↓",
    "mixed_words": "Mixed words ↓",
    "reference_words_removed": "Reference words removed ↑",
    "reduction": "Reduction ↑",
})
summary["Reduction ↑"] = summary["Reduction ↑"].map(lambda value: f"{value:.1%}")
summary = summary.round({
    "Original words ↓": 2,
    "Mixed words ↓": 2,
    "Reference words removed ↑": 2,
})
display(summary)

In [ ]:
# Show group sizes so the summary can be interpreted alongside its sample counts.
display(data.groupby("Benchmark", sort=False).size().rename("N").reindex(benchmark_order).dropna().astype(int).to_frame())